# 1\.5\.5 Engineer Variability and Deviation Features

Purpose: create features that measure how unusual a mobility state is, without assigning final anomaly labels yet\.

We start by loading the completed 1\.5\.4 decomposition feature panel\. This notebook uses that panel as its feature base and adds anomaly\-severity features without changing the Taxi Zone × Date × Temporal Bucket grain\.

In [1]:
# ---------------------------------------------------------------------
# Imports and project paths
# ---------------------------------------------------------------------

from pathlib import Path
import warnings
import gc
import time

import duckdb
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)

PIPELINE_DATA_DIR = Path("pipeline_data")

INPUT_DIR = PIPELINE_DATA_DIR / "1.5.4.final_tables"
OUTPUT_DIR = PIPELINE_DATA_DIR / "1.5.5.final_tables"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

INPUT_PANEL_PATH = INPUT_DIR / "temporal_decomposition_feature_panel.parquet"
INPUT_MANIFEST_PATH = INPUT_DIR / "temporal_decomposition_feature_manifest.csv"
INPUT_QA_PATH = INPUT_DIR / "temporal_decomposition_feature_qa_summary.csv"

OUTPUT_PANEL_PATH = OUTPUT_DIR / "variability_anomaly_feature_panel.parquet"
OUTPUT_MANIFEST_PATH = OUTPUT_DIR / "variability_anomaly_feature_manifest.csv"
OUTPUT_QA_PATH = OUTPUT_DIR / "variability_anomaly_feature_qa_summary.csv"

WRITE_OUTPUTS = True

required_input_paths = {
    "1.5.4 decomposition feature panel": INPUT_PANEL_PATH,
    "1.5.4 decomposition feature manifest": INPUT_MANIFEST_PATH,
    "1.5.4 decomposition QA summary": INPUT_QA_PATH,
}

for label, path in required_input_paths.items():
    print(f"{label}: {path.exists()} | {path}")

missing_input_paths = [
    path
    for path in required_input_paths.values()
    if not path.exists()
]

assert not missing_input_paths, (
    f"Missing required 1.5.5 input files: {missing_input_paths}"
)

1.5.4 decomposition feature panel: True | pipeline_data/1.5.4.final_tables/temporal_decomposition_feature_panel.parquet
1.5.4 decomposition feature manifest: True | pipeline_data/1.5.4.final_tables/temporal_decomposition_feature_manifest.csv
1.5.4 decomposition QA summary: True | pipeline_data/1.5.4.final_tables/temporal_decomposition_feature_qa_summary.csv


In [2]:
# ---------------------------------------------------------------------
# 1.5.5 intermediate output controls
# ---------------------------------------------------------------------

INTERMEDIATE_OUTPUT_DIR = PIPELINE_DATA_DIR / "1.5.5.intermediate_tables"
INTERMEDIATE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

REBUILD_ZSCORE_ANOMALY_FEATURES = False
REBUILD_PERCENTILE_ANOMALY_FEATURES = False

ZSCORE_ANOMALY_FEATURE_PATH = (
    INTERMEDIATE_OUTPUT_DIR / "zscore_anomaly_severity_features.parquet"
)

ZSCORE_ANOMALY_SUMMARY_PATH = (
    INTERMEDIATE_OUTPUT_DIR / "zscore_anomaly_severity_feature_summary.csv"
)

PERCENTILE_ANOMALY_FEATURE_PATH = (
    INTERMEDIATE_OUTPUT_DIR / "percentile_anomaly_severity_features.parquet"
)

PERCENTILE_ANOMALY_SUMMARY_PATH = (
    INTERMEDIATE_OUTPUT_DIR / "percentile_anomaly_severity_feature_summary.csv"
)

Next, we define the analytical grain, core metrics, and required 1\.5\.4 decomposition features that this notebook will build on\.

In [3]:
# ---------------------------------------------------------------------
# Analytical grain and required input columns
# ---------------------------------------------------------------------

GRAIN_COLUMNS = [
    "taxi_zone_id",
    "date",
    "temporal_bucket",
]

BASELINE_GROUP_COLUMNS = [
    "taxi_zone_id",
    "temporal_bucket",
    "pre_post_cp",
]

METADATA_COLUMNS = [
    "taxi_zone_id",
    "zone",
    "borough",
    "canonical_location_id",
    "date",
    "year",
    "month",
    "day_of_week",
    "temporal_bucket",
    "pre_post_cp",
]

# 1.5.5 scores the observed core mobility metrics against local history.
# The 1.5.4 residual/Fourier columns are required input context, but they
# are not recreated or expanded in this notebook.
ANOMALY_SEVERITY_METRICS = [
    "bus_trip_count",
    "avg_bus_speed",
    "subway_ridership",
    "taxi_trip_count",
    "taxi_avg_trip_speed",
    "fhvhv_trip_count",
    "fhvhv_avg_trip_speed",
]

REQUIRED_STL_RESIDUAL_COLUMNS = [
    f"{metric}_residual"
    for metric in ANOMALY_SEVERITY_METRICS
]

REQUIRED_FOURIER_RESIDUAL_COLUMNS = [
    f"{metric}_fourier20_residual_reconstructed"
    for metric in ANOMALY_SEVERITY_METRICS
]

ANOMALY_SEVERITY_FEATURE_SUFFIXES = [
    "deviation_zscore",
    "deviation_abs_zscore",
    "deviation_percentile_rank",
]

ANOMALY_SEVERITY_FEATURE_COLUMNS = [
    f"{metric}_{suffix}"
    for metric in ANOMALY_SEVERITY_METRICS
    for suffix in ANOMALY_SEVERITY_FEATURE_SUFFIXES
]

print(f"Anomaly-severity metrics: {len(ANOMALY_SEVERITY_METRICS)}")
print(f"Planned anomaly-severity feature count: {len(ANOMALY_SEVERITY_FEATURE_COLUMNS)}")

Anomaly-severity metrics: 7
Planned anomaly-severity feature count: 21


## 1\.5\.5\.1 Load and Validate Inputs

Now we load the panel and run a compact readiness check\. The goal here is not to re\-audit 1\.5\.4, but to confirm that the input is ready for anomaly\-severity feature engineering\.

In [4]:
# ---------------------------------------------------------------------
# Load 1.5.4 panel and validate readiness
# ---------------------------------------------------------------------

anomaly_df = pd.read_parquet(INPUT_PANEL_PATH)
anomaly_df["date"] = pd.to_datetime(anomaly_df["date"])

required_columns = sorted(
    set(
        GRAIN_COLUMNS
        + BASELINE_GROUP_COLUMNS
        + METADATA_COLUMNS
        + ANOMALY_SEVERITY_METRICS
        + REQUIRED_STL_RESIDUAL_COLUMNS
        + REQUIRED_FOURIER_RESIDUAL_COLUMNS
    )
)

missing_required_columns = [
    column
    for column in required_columns
    if column not in anomaly_df.columns
]

duplicate_grain_rows = anomaly_df.duplicated(GRAIN_COLUMNS).sum()

input_readiness_summary_df = pd.DataFrame(
    [
        {
            "qa_check": "row_count",
            "value": len(anomaly_df),
            "status": "pass" if len(anomaly_df) > 0 else "fail",
        },
        {
            "qa_check": "column_count",
            "value": anomaly_df.shape[1],
            "status": "pass",
        },
        {
            "qa_check": "date_range",
            "value": f"{anomaly_df['date'].min().date()} to {anomaly_df['date'].max().date()}",
            "status": "pass",
        },
        {
            "qa_check": "unique_taxi_zones",
            "value": anomaly_df["taxi_zone_id"].nunique(),
            "status": "pass",
        },
        {
            "qa_check": "unique_temporal_buckets",
            "value": anomaly_df["temporal_bucket"].nunique(),
            "status": "pass",
        },
        {
            "qa_check": "duplicate_grain_rows",
            "value": duplicate_grain_rows,
            "status": "pass" if duplicate_grain_rows == 0 else "fail",
        },
        {
            "qa_check": "missing_required_columns",
            "value": len(missing_required_columns),
            "status": "pass" if len(missing_required_columns) == 0 else "fail",
        },
    ]
)

input_feature_inventory_df = pd.DataFrame(
    [
        {
            "feature_group": "core anomaly-severity metrics",
            "expected_columns": len(ANOMALY_SEVERITY_METRICS),
            "available_columns": sum(
                column in anomaly_df.columns
                for column in ANOMALY_SEVERITY_METRICS
            ),
        },
        {
            "feature_group": "1.5.4 STL residual features",
            "expected_columns": len(REQUIRED_STL_RESIDUAL_COLUMNS),
            "available_columns": sum(
                column in anomaly_df.columns
                for column in REQUIRED_STL_RESIDUAL_COLUMNS
            ),
        },
        {
            "feature_group": "1.5.4 Fourier residual reconstruction features",
            "expected_columns": len(REQUIRED_FOURIER_RESIDUAL_COLUMNS),
            "available_columns": sum(
                column in anomaly_df.columns
                for column in REQUIRED_FOURIER_RESIDUAL_COLUMNS
            ),
        },
    ]
)

input_feature_inventory_df["status"] = np.where(
    input_feature_inventory_df["expected_columns"]
    == input_feature_inventory_df["available_columns"],
    "pass",
    "fail",
)

display(input_readiness_summary_df)
display(input_feature_inventory_df)

assert len(anomaly_df) > 0, "1.5.4 input panel is empty."

assert duplicate_grain_rows == 0, (
    "Duplicate rows detected at the Taxi Zone × Date × Temporal Bucket grain."
)

assert not missing_required_columns, (
    f"Missing required 1.5.5 input columns: {missing_required_columns}"
)

assert (
    input_feature_inventory_df["status"].eq("pass").all()
), (
    "One or more required 1.5.4 feature groups is incomplete."
)

print("1.5.5 input panel loaded and ready for anomaly-severity feature engineering.")

,qa_check,value,status
0,row_count,1559590,pass
1,column_count,321,pass
2,date_range,2023-01-01 to 2026-03-31,pass
3,unique_taxi_zones,263,pass
4,unique_temporal_buckets,10,pass
5,duplicate_grain_rows,0,pass
6,missing_required_columns,0,pass


,feature_group,expected_columns,available_columns,status
0,core anomaly-severity metrics,7,7,pass
1,1.5.4 STL residual features,7,7,pass
2,1.5.4 Fourier residual reconstruction features,7,7,pass


1.5.5 input panel loaded and ready for anomaly-severity feature engineering.


Findings\. The 1\.5\.4 decomposition feature panel loaded successfully and preserved the Taxi Zone × Date × Temporal Bucket grain\. The seven anomaly\-severity metrics and required 1\.5\.4 residual features are present, so the panel is ready for retrospective baseline construction\.

We also define the baseline strategy here so the rest of the notebook has one clear reference point\. These anomaly\-severity features use full\-history retrospective baselines grouped by Taxi Zone × Temporal Bucket × congestion\-pricing regime\. This means each observation is scored against mobility behavior from the same zone, same daypart bucket, and same pre/post policy period\. These features measure deviation severity only; final anomaly labels are created later in the anomaly\-detection notebooks\. Future split\-safe baselines should respect the January 5, 2025 policy break, including pre\-CP model\-development splits and separate post\-CP evaluation\.

In [5]:
# ---------------------------------------------------------------------
# Configure anomaly-severity baseline strategy
# ---------------------------------------------------------------------

MIN_BASELINE_OBSERVATIONS = 10
ZSCORE_EPSILON = 1e-6

# Current 1.5.5 production features use full-history retrospective baselines.
# The commented dates below are planning notes for future split-safe modeling
# and supervised evaluation; they are intentionally not used in this notebook.
#
# Congestion pricing may have changed NYC mobility dynamics, so future
# model-development splits should respect the policy break.
#
# Pre-CP model-development window:
# Use this window to train/validate/test models on the pre-policy regime.
# PRE_CP_MODEL_START_DATE = "2023-01-01"
# PRE_CP_MODEL_END_DATE = "2025-01-04"
#
# Example future pre-CP split:
# PRE_CP_TRAIN_END_DATE = "2024-06-30"
# PRE_CP_VALIDATION_START_DATE = "2024-07-01"
# PRE_CP_VALIDATION_END_DATE = "2024-10-31"
# PRE_CP_TEST_START_DATE = "2024-11-01"
# PRE_CP_TEST_END_DATE = "2025-01-04"
#
# Post-CP evaluation window:
# Use this window to evaluate policy-era behavior separately from pre-CP
# model development.
# CONGESTION_PRICING_START_DATE = "2025-01-05"
# POST_CP_EVALUATION_START_DATE = "2025-01-05"
# POST_CP_EVALUATION_END_DATE = "2026-03-31"

baseline_strategy_summary_df = pd.DataFrame(
    [
        {
            "setting": "baseline_grain",
            "value": " × ".join(BASELINE_GROUP_COLUMNS),
        },
        {
            "setting": "baseline_type",
            "value": "full-history retrospective",
        },
        {
            "setting": "feature_role",
            "value": "deviation severity, not final anomaly labels",
        },
        {
            "setting": "production_feature_types",
            "value": ", ".join(ANOMALY_SEVERITY_FEATURE_SUFFIXES),
        },
        {
            "setting": "metrics",
            "value": len(ANOMALY_SEVERITY_METRICS),
        },
        {
            "setting": "planned_feature_count",
            "value": len(ANOMALY_SEVERITY_FEATURE_COLUMNS),
        },
        {
            "setting": "minimum_baseline_observations",
            "value": MIN_BASELINE_OBSERVATIONS,
        },
    ]
)

display(baseline_strategy_summary_df)

assert len(ANOMALY_SEVERITY_FEATURE_COLUMNS) == (
    len(ANOMALY_SEVERITY_METRICS)
    * len(ANOMALY_SEVERITY_FEATURE_SUFFIXES)
), (
    "Planned anomaly-severity feature count does not match metric × suffix configuration."
)

print(
    "Anomaly-severity baseline strategy configured. "
    f"Planned features: {len(ANOMALY_SEVERITY_FEATURE_COLUMNS):,}."
)

,setting,value
0,baseline_grain,taxi_zone_id × temporal_bucket × pre_post_cp
1,baseline_type,full-history retrospective
2,feature_role,"deviation severity, not final anomaly labels"
3,production_feature_types,"deviation_zscore, deviation_abs_zscore, deviat..."
4,metrics,7
5,planned_feature_count,21
6,minimum_baseline_observations,10


Anomaly-severity baseline strategy configured. Planned features: 21.


Findings\. The baseline strategy is set up to score each observation against its own Taxi Zone × Temporal Bucket × congestion\-pricing\-regime history\. This gives us 21 planned anomaly\-severity features across the seven core metrics: z\-score, absolute z\-score, and percentile rank for each metric\. These features measure how unusual a mobility state looks relative to its local historical context, but they do not assign final anomaly labels\.

## 1\.5\.5\.2 Build Historical Baselines

We first build the historical reference tables that will define “usual” mobility behavior for each Taxi Zone × Temporal Bucket × congestion\-pricing regime\. These baselines are retrospective and descriptive: they summarize the observed history available inside each same\-bucket series, then the next section uses them to score how unusual each observation looks relative to that local context\.

### Build Metric\-Level Baseline Statistics

First, we calculate baseline statistics for each of the seven anomaly\-severity metrics\. These statistics give us the mean and standard deviation needed for z\-scores, along with compact coverage and distribution checks for each baseline group\.

In [6]:
# ---------------------------------------------------------------------
# Build metric-level baseline statistics
# ---------------------------------------------------------------------

baseline_stat_records = []

for metric in ANOMALY_SEVERITY_METRICS:
    # Each metric gets its own baseline table because the observed values,
    # null patterns, and distribution shapes differ by transportation mode.
    metric_baseline_df = (
        anomaly_df
        .groupby(BASELINE_GROUP_COLUMNS, observed=True)
        .agg(
            baseline_observations=(metric, "count"),
            baseline_mean=(metric, "mean"),
            baseline_std=(metric, "std"),
            baseline_min=(metric, "min"),
            baseline_median=(metric, "median"),
            baseline_max=(metric, "max"),
        )
        .reset_index()
    )

    metric_baseline_df["metric"] = metric

    # Groups below this threshold are retained for QA, but they should not
    # be treated as reliable baselines when generating deviation features.
    metric_baseline_df["baseline_ready"] = (
        metric_baseline_df["baseline_observations"]
        >= MIN_BASELINE_OBSERVATIONS
    )

    baseline_stat_records.append(metric_baseline_df)

baseline_statistics_df = (
    pd.concat(baseline_stat_records, ignore_index=True)
    .loc[
        :,
        BASELINE_GROUP_COLUMNS
        + [
            "metric",
            "baseline_observations",
            "baseline_mean",
            "baseline_std",
            "baseline_min",
            "baseline_median",
            "baseline_max",
            "baseline_ready",
        ],
    ]
)

# This compact summary tells us whether each metric has enough historical
# support for z-score style deviation features.
baseline_statistics_summary_df = (
    baseline_statistics_df
    .groupby("metric", as_index=False)
    .agg(
        baseline_groups=("baseline_ready", "size"),
        ready_baseline_groups=("baseline_ready", "sum"),
        min_observations=("baseline_observations", "min"),
        median_observations=("baseline_observations", "median"),
        max_observations=("baseline_observations", "max"),
        min_baseline_std=("baseline_std", "min"),
        median_baseline_std=("baseline_std", "median"),
        zero_or_null_std_groups=(
            "baseline_std",
            lambda series: (
                series.isna()
                | series.le(ZSCORE_EPSILON)
            ).sum(),
        ),
    )
)

baseline_statistics_summary_df["ready_baseline_group_pct"] = (
    baseline_statistics_summary_df["ready_baseline_groups"]
    / baseline_statistics_summary_df["baseline_groups"]
)

numeric_summary_columns = [
    "median_observations",
    "min_baseline_std",
    "median_baseline_std",
    "ready_baseline_group_pct",
]

baseline_statistics_summary_df[numeric_summary_columns] = (
    baseline_statistics_summary_df[numeric_summary_columns]
    .round(4)
)

display(baseline_statistics_summary_df)

assert not baseline_statistics_df.empty, (
    "Baseline statistics table is empty."
)

assert (
    baseline_statistics_df["baseline_ready"]
    .any()
), (
    "No baseline groups meet the minimum observation threshold."
)

assert (
    baseline_statistics_summary_df["ready_baseline_groups"]
    .gt(0)
    .all()
), (
    "At least one metric has no ready baseline groups."
)

print(
    "Metric-level baseline statistics created. "
    f"Rows: {len(baseline_statistics_df):,}."
)

,metric,baseline_groups,ready_baseline_groups,min_observations,median_observations,max_observations,min_baseline_std,median_baseline_std,zero_or_null_std_groups,ready_baseline_group_pct
0,avg_bus_speed,5260,4980,0,210.0,525,0.0933,0.3538,279,0.9468
1,bus_trip_count,5260,4980,0,210.0,525,0.0000,78.7053,281,0.9468
2,fhvhv_avg_trip_speed,5260,5260,129,266.0,525,0.0000,1.0648,23,1.0000
3,fhvhv_trip_count,5260,5260,129,266.0,525,0.0000,72.1995,23,1.0000
4,subway_ridership,5260,3120,0,129.0,525,0.0000,407.1914,2170,0.5932
5,taxi_avg_trip_speed,5260,4964,0,183.0,525,0.0081,4.4189,90,0.9437
6,taxi_trip_count,5260,5260,129,266.0,525,0.0000,3.0162,47,1.0000


Metric-level baseline statistics created. Rows: 36,820.


Findings\. The baseline\-statistics table is ready for feature generation\. It contains 36,820 metric\-level baseline rows, with 5,260 baseline groups per metric\. Taxi trip count, FHVHV trip count, and FHVHV average trip speed have complete ready\-baseline coverage; Bus trip count, average bus speed, and Taxi average trip speed are also strong at roughly 94–95%\. Subway ridership has 3,120 ready groups out of 5,260, or 59\.3%, which matches the project’s known subway geography: subway coverage exists in 156 of 263 Taxi Zones, so many zone\-level subway baselines are naturally unavailable rather than missing because of a processing issue\.

We also preview a few baseline rows separately\. This keeps the main summary readable while still showing what a usable baseline group looks like and what a not\-ready group looks like\.

In [7]:
# ---------------------------------------------------------------------
# Preview baseline statistic examples
# ---------------------------------------------------------------------

# Show usable and not-ready groups separately so the preview does not get
# dominated by special TLC zones 264 and 265.
baseline_statistics_preview_source_df = (
    baseline_statistics_df
    .loc[~baseline_statistics_df["taxi_zone_id"].isin([264, 265])]
    .copy()
)

baseline_statistics_preview_df = pd.concat(
    [
        baseline_statistics_preview_source_df
        .loc[baseline_statistics_preview_source_df["baseline_ready"]]
        .tail(5)
        .assign(example_type="ready baseline"),
        baseline_statistics_preview_source_df
        .loc[~baseline_statistics_preview_source_df["baseline_ready"]]
        .tail(5)
        .assign(example_type="not ready baseline"),
    ],
    ignore_index=True,
)

preview_column_order = [
    "example_type",
    "taxi_zone_id",
    "temporal_bucket",
    "pre_post_cp",
    "metric",
    "baseline_observations",
    "baseline_mean",
    "baseline_std",
    "baseline_ready",
]

baseline_statistics_preview_df = baseline_statistics_preview_df[preview_column_order]

numeric_preview_columns = [
    "baseline_mean",
    "baseline_std",
]

baseline_statistics_preview_df[numeric_preview_columns] = (
    baseline_statistics_preview_df[numeric_preview_columns]
    .round(4)
)

display(baseline_statistics_preview_df)

,example_type,taxi_zone_id,temporal_bucket,pre_post_cp,metric,baseline_observations,baseline_mean,baseline_std,baseline_ready
0,ready baseline,263,weekend_midday,pre_cp,fhvhv_avg_trip_speed,210,14.5845,1.2645,True
1,ready baseline,263,weekend_overnight,post_cp,fhvhv_avg_trip_speed,129,18.3589,0.8730,True
2,ready baseline,263,weekend_overnight,pre_cp,fhvhv_avg_trip_speed,210,18.8582,0.8508,True
3,ready baseline,263,weekend_pm_peak,post_cp,fhvhv_avg_trip_speed,129,13.2449,1.4567,True
4,ready baseline,263,weekend_pm_peak,pre_cp,fhvhv_avg_trip_speed,210,13.3990,1.2718,True
5,not ready baseline,251,weekend_midday,post_cp,taxi_avg_trip_speed,5,24.6504,5.7854,False
6,not ready baseline,251,weekend_midday,pre_cp,taxi_avg_trip_speed,6,20.3808,12.6893,False
7,not ready baseline,251,weekend_overnight,pre_cp,taxi_avg_trip_speed,7,23.5002,5.8872,False
8,not ready baseline,253,weekend_am_peak,post_cp,taxi_avg_trip_speed,7,23.9423,6.3264,False
9,not ready baseline,253,weekend_am_peak,pre_cp,taxi_avg_trip_speed,6,28.4389,7.3136,False


Findings\. The preview confirms how the readiness flag works at the row level\. Ready baseline rows have enough historical observations to support deviation scoring, while not\-ready rows fall below the 10\-observation threshold and are kept for QA rather than treated as reliable baselines\. This is also why the preview is separate from the main summary table: the row examples are useful for checking the structure, but the metric\-level summary is the real decision table\.

### Build Percentile Reference Tables

Percentile ranks use the same local history as the z\-score baselines: Taxi Zone × Temporal Bucket × congestion\-pricing regime\. Here we create a compact reference table that tells us which baseline groups have enough observations to support percentile scoring\. We do not need to materialize a giant row\-level lookup table here; the actual percentile\-rank feature can be calculated directly during feature generation\.

In [8]:
# ---------------------------------------------------------------------
# Build percentile reference tables
# ---------------------------------------------------------------------

# Percentile ranks use the same baseline groups as z-scores, but they only
# need an empirical reference distribution rather than mean/std statistics.
percentile_reference_df = (
    baseline_statistics_df[
        BASELINE_GROUP_COLUMNS
        + [
            "metric",
            "baseline_observations",
            "baseline_min",
            "baseline_median",
            "baseline_max",
            "baseline_ready",
        ]
    ]
    .rename(
        columns={
            "baseline_observations": "reference_observations",
            "baseline_min": "reference_min",
            "baseline_median": "reference_median",
            "baseline_max": "reference_max",
            "baseline_ready": "percentile_reference_ready",
        }
    )
    .copy()
)

# Keep the readiness rule aligned with the z-score baseline rule so the
# anomaly-severity feature family has one consistent baseline standard.
percentile_reference_df["percentile_reference_ready"] = (
    percentile_reference_df["reference_observations"]
    >= MIN_BASELINE_OBSERVATIONS
)

# This metric-level summary shows whether percentile scoring is broadly
# supported across the seven production metrics.
percentile_reference_summary_df = (
    percentile_reference_df
    .groupby("metric", as_index=False)
    .agg(
        reference_groups=("percentile_reference_ready", "size"),
        ready_reference_groups=("percentile_reference_ready", "sum"),
        zero_reference_groups=(
            "reference_observations",
            lambda series: series.eq(0).sum(),
        ),
        min_reference_observations=("reference_observations", "min"),
        median_reference_observations=("reference_observations", "median"),
        max_reference_observations=("reference_observations", "max"),
    )
)

percentile_reference_summary_df["ready_reference_group_pct"] = (
    percentile_reference_summary_df["ready_reference_groups"]
    / percentile_reference_summary_df["reference_groups"]
)

percentile_reference_summary_df[
    [
        "median_reference_observations",
        "ready_reference_group_pct",
    ]
] = (
    percentile_reference_summary_df[
        [
            "median_reference_observations",
            "ready_reference_group_pct",
        ]
    ]
    .round(4)
)

display(percentile_reference_summary_df)

assert not percentile_reference_df.empty, (
    "Percentile reference table is empty."
)

assert percentile_reference_df["metric"].nunique() == len(ANOMALY_SEVERITY_METRICS), (
    "Percentile reference table does not cover all anomaly-severity metrics."
)

assert (
    percentile_reference_summary_df["ready_reference_groups"]
    .gt(0)
    .all()
), (
    "At least one metric has no ready percentile reference groups."
)

print(
    "Percentile reference table created. "
    f"Rows: {len(percentile_reference_df):,}."
)

,metric,reference_groups,ready_reference_groups,zero_reference_groups,min_reference_observations,median_reference_observations,max_reference_observations,ready_reference_group_pct
0,avg_bus_speed,5260,4980,274,0,210.0,525,0.9468
1,bus_trip_count,5260,4980,274,0,210.0,525,0.9468
2,fhvhv_avg_trip_speed,5260,5260,0,129,266.0,525,1.0000
3,fhvhv_trip_count,5260,5260,0,129,266.0,525,1.0000
4,subway_ridership,5260,3120,2140,0,129.0,525,0.5932
5,taxi_avg_trip_speed,5260,4964,47,0,183.0,525,0.9437
6,taxi_trip_count,5260,5260,0,129,266.0,525,1.0000


Percentile reference table created. Rows: 36,820.


Findings\. The percentile reference table mirrors the baseline\-statistics coverage because it uses the same Taxi Zone × Temporal Bucket × congestion\-pricing\-regime groups and the same minimum\-observation rule\. The strongest coverage remains Taxi trip count and both FHVHV metrics at 100% ready reference groups, while Bus and Taxi average speed are still around 94–95%\. Subway ridership has 3,120 ready reference groups out of 5,260, or 59\.3%, matching the 156 of 263 Taxi Zones with subway coverage\. For zones without subway service, percentile\-rank features should remain unavailable rather than being forced into artificial values\.

We preview percentile reference rows separately for the same reason: the summary tells us whether the reference tables are usable overall, while the preview shows how ready and not\-ready groups are represented\.

In [9]:
# ---------------------------------------------------------------------
# Preview percentile reference examples
# ---------------------------------------------------------------------

# Show usable and not-ready percentile references separately so the preview
# is informative without leaning on special TLC zones 264 and 265.
percentile_reference_preview_source_df = (
    percentile_reference_df
    .loc[~percentile_reference_df["taxi_zone_id"].isin([264, 265])]
    .copy()
)

percentile_reference_preview_df = pd.concat(
    [
        percentile_reference_preview_source_df
        .loc[percentile_reference_preview_source_df["percentile_reference_ready"]]
        .tail(5)
        .assign(example_type="ready reference"),
        percentile_reference_preview_source_df
        .loc[~percentile_reference_preview_source_df["percentile_reference_ready"]]
        .tail(5)
        .assign(example_type="not ready reference"),
    ],
    ignore_index=True,
)

preview_column_order = [
    "example_type",
    "taxi_zone_id",
    "temporal_bucket",
    "pre_post_cp",
    "metric",
    "reference_observations",
    "reference_min",
    "reference_median",
    "reference_max",
    "percentile_reference_ready",
]

percentile_reference_preview_df = percentile_reference_preview_df[preview_column_order]

numeric_preview_columns = [
    "reference_min",
    "reference_median",
    "reference_max",
]

percentile_reference_preview_df[numeric_preview_columns] = (
    percentile_reference_preview_df[numeric_preview_columns]
    .round(4)
)

display(percentile_reference_preview_df)

,example_type,taxi_zone_id,temporal_bucket,pre_post_cp,metric,reference_observations,reference_min,reference_median,reference_max,percentile_reference_ready
0,ready reference,263,weekend_midday,pre_cp,fhvhv_avg_trip_speed,210,10.3663,14.6872,18.6336,True
1,ready reference,263,weekend_overnight,post_cp,fhvhv_avg_trip_speed,129,15.0014,18.4259,19.9374,True
2,ready reference,263,weekend_overnight,pre_cp,fhvhv_avg_trip_speed,210,16.7736,18.8591,22.0680,True
3,ready reference,263,weekend_pm_peak,post_cp,fhvhv_avg_trip_speed,129,9.3488,13.0915,16.4408,True
4,ready reference,263,weekend_pm_peak,pre_cp,fhvhv_avg_trip_speed,210,10.2723,13.4528,16.0344,True
5,not ready reference,251,weekend_midday,post_cp,taxi_avg_trip_speed,5,17.7632,24.3227,33.4813,False
6,not ready reference,251,weekend_midday,pre_cp,taxi_avg_trip_speed,6,3.4395,23.9446,35.5674,False
7,not ready reference,251,weekend_overnight,pre_cp,taxi_avg_trip_speed,7,15.0829,26.5234,30.4972,False
8,not ready reference,253,weekend_am_peak,post_cp,taxi_avg_trip_speed,7,12.9897,25.2401,32.6868,False
9,not ready reference,253,weekend_am_peak,pre_cp,taxi_avg_trip_speed,6,20.4658,27.4498,41.4618,False


Findings\. The preview shows the difference between usable and not\-ready percentile references\. Ready reference rows have much larger historical samples, such as 129 or 210 observations, while not\-ready examples have only 5–7 observations and are correctly marked as unavailable for percentile scoring\. The reference min, median, and max values are still retained for inspection, but the readiness flag is the important field because it prevents thin histories from being treated like stable local baselines\.

### Section Close

The historical baseline tables are ready for anomaly\-severity feature generation\. The z\-score and percentile\-reference baselines use the same local comparison grain: Taxi Zone × Temporal Bucket × congestion\-pricing regime\. Most metrics have strong baseline coverage, and Subway’s lower coverage is expected because subway service exists in 156 of 263 Taxi Zones\. These baselines give the next section the reference values needed to calculate z\-scores, absolute z\-scores, and percentile ranks without changing the panel grain\.

## 1\.5\.5\.3 Generate Anomaly\-Severity Features

This section turns the historical baselines into anomaly\-severity features\. For each core mobility metric, we score the observed value against its own Taxi Zone × Temporal Bucket × congestion\-pricing\-regime history\. The features created here do not label anomalies; they measure how far each observation sits from its local historical baseline\.

### Generate Z\-Score and Absolute Z\-Score Features

We start with the most direct anomaly\-severity measure: standardized deviation from the local historical mean\. The z\-score keeps direction, while the absolute z\-score captures severity regardless of whether the observation is unusually high or unusually low\.

In [10]:
# ---------------------------------------------------------------------
# Generate z-score and absolute z-score anomaly-severity features
# ---------------------------------------------------------------------

zscore_start_time = time.perf_counter()

if ZSCORE_ANOMALY_FEATURE_PATH.exists() and not REBUILD_ZSCORE_ANOMALY_FEATURES:
    anomaly_severity_feature_df = pd.read_parquet(ZSCORE_ANOMALY_FEATURE_PATH)
    zscore_feature_summary_df = pd.read_csv(ZSCORE_ANOMALY_SUMMARY_PATH)

    print(
        "Loaded cached z-score anomaly-severity features: "
        f"{ZSCORE_ANOMALY_FEATURE_PATH}"
    )

else:
    # Keep the feature output skinny. Copying the full 1.5M-row panel would
    # create unnecessary memory pressure.
    anomaly_severity_feature_df = anomaly_df[GRAIN_COLUMNS].copy()
    anomaly_severity_feature_df["_row_id"] = np.arange(
        len(anomaly_severity_feature_df)
    )

    zscore_feature_records = []

    for metric in ANOMALY_SEVERITY_METRICS:
        metric_start_time = time.perf_counter()

        # Work with only the columns needed for this metric.
        metric_work_df = (
            anomaly_df[
                [
                    "taxi_zone_id",
                    "date",
                    "temporal_bucket",
                    "pre_post_cp",
                    metric,
                ]
            ]
            .copy()
        )

        metric_work_df["_row_id"] = np.arange(len(metric_work_df))

        metric_baseline_df = (
            baseline_statistics_df
            .loc[
                baseline_statistics_df["metric"] == metric,
                BASELINE_GROUP_COLUMNS
                + [
                    "baseline_observations",
                    "baseline_mean",
                    "baseline_std",
                    "baseline_ready",
                ],
            ]
            .copy()
        )

        metric_work_df = metric_work_df.merge(
            metric_baseline_df,
            on=BASELINE_GROUP_COLUMNS,
            how="left",
            validate="many_to_one",
        )

        zscore_column = f"{metric}_deviation_zscore"
        abs_zscore_column = f"{metric}_deviation_abs_zscore"

        ready_mask = (
            metric_work_df["baseline_ready"].fillna(False)
            & metric_work_df[metric].notna()
            & metric_work_df["baseline_std"].notna()
            & metric_work_df["baseline_std"].gt(ZSCORE_EPSILON)
        )

        metric_work_df[zscore_column] = np.nan

        metric_work_df.loc[ready_mask, zscore_column] = (
            (
                metric_work_df.loc[ready_mask, metric]
                - metric_work_df.loc[ready_mask, "baseline_mean"]
            )
            / metric_work_df.loc[ready_mask, "baseline_std"]
        )

        metric_work_df[abs_zscore_column] = (
            metric_work_df[zscore_column]
            .abs()
        )

        anomaly_severity_feature_df = anomaly_severity_feature_df.merge(
            metric_work_df[
                [
                    "_row_id",
                    zscore_column,
                    abs_zscore_column,
                ]
            ],
            on="_row_id",
            how="left",
            validate="one_to_one",
        )

        zscore_feature_records.append(
            {
                "metric": metric,
                "ready_baseline_groups": metric_baseline_df[
                    "baseline_ready"
                ].sum(),
                "zscore_non_null_observations": metric_work_df[
                    zscore_column
                ].notna().sum(),
                "zscore_null_pct": metric_work_df[
                    zscore_column
                ].isna().mean(),
                "mean_abs_zscore": metric_work_df[
                    abs_zscore_column
                ].mean(),
                "max_abs_zscore": metric_work_df[
                    abs_zscore_column
                ].max(),
                "runtime_seconds": time.perf_counter() - metric_start_time,
            }
        )

        print(
            f"Created z-score features for {metric} "
            f"in {time.perf_counter() - metric_start_time:.2f} seconds."
        )

        del metric_work_df, metric_baseline_df, ready_mask
        gc.collect()

    zscore_feature_summary_df = pd.DataFrame(zscore_feature_records)

    zscore_feature_summary_df[
        [
            "zscore_null_pct",
            "mean_abs_zscore",
            "max_abs_zscore",
            "runtime_seconds",
        ]
    ] = (
        zscore_feature_summary_df[
            [
                "zscore_null_pct",
                "mean_abs_zscore",
                "max_abs_zscore",
                "runtime_seconds",
            ]
        ]
        .round(4)
    )

    anomaly_severity_feature_df.to_parquet(
        ZSCORE_ANOMALY_FEATURE_PATH,
        index=False,
    )

    zscore_feature_summary_df.to_csv(
        ZSCORE_ANOMALY_SUMMARY_PATH,
        index=False,
    )

    print(
        "Saved z-score anomaly-severity features: "
        f"{ZSCORE_ANOMALY_FEATURE_PATH}"
    )

display(zscore_feature_summary_df)

expected_zscore_columns = [
    f"{metric}_deviation_zscore"
    for metric in ANOMALY_SEVERITY_METRICS
]

expected_abs_zscore_columns = [
    f"{metric}_deviation_abs_zscore"
    for metric in ANOMALY_SEVERITY_METRICS
]

missing_zscore_columns = [
    column
    for column in expected_zscore_columns + expected_abs_zscore_columns
    if column not in anomaly_severity_feature_df.columns
]

assert not missing_zscore_columns, (
    f"Missing generated z-score feature columns: {missing_zscore_columns}"
)

assert (
    zscore_feature_summary_df["zscore_non_null_observations"]
    .gt(0)
    .all()
), (
    "At least one metric produced no non-null z-score values."
)

print(
    "Z-score anomaly-severity feature step complete. "
    f"Runtime: {time.perf_counter() - zscore_start_time:.2f} seconds."
)

Loaded cached z-score anomaly-severity features: pipeline_data/1.5.5.intermediate_tables/zscore_anomaly_severity_features.parquet


,metric,ready_baseline_groups,zscore_non_null_observations,zscore_null_pct,mean_abs_zscore,max_abs_zscore,runtime_seconds
0,bus_trip_count,4980,1476448,0.0533,0.8640,7.2295,1.3803
1,avg_bus_speed,4980,1476465,0.0533,0.8017,8.2588,1.1274
2,subway_ridership,3120,918315,0.4112,0.7108,22.2995,1.1584
3,taxi_trip_count,5260,1547183,0.0080,0.7048,22.8692,1.1894
4,taxi_avg_trip_speed,4964,1124054,0.2793,0.7512,14.6024,1.2798
5,fhvhv_trip_count,5260,1553375,0.0040,0.7312,19.5110,1.2526
6,fhvhv_avg_trip_speed,5260,1553375,0.0040,0.7550,20.5910,1.2723


Z-score anomaly-severity feature step complete. Runtime: 1.09 seconds.


Findings\. The z\-score feature step completed cleanly and quickly, generating 14 features across the seven core metrics in about 14 seconds\. Coverage follows the baseline availability we already saw: FHVHV and Taxi trip counts are nearly complete, Bus features have about 5\.3% nulls, and Subway has the highest null rate at 41\.1% because subway service only exists in 156 of 263 Taxi Zones\. Taxi average trip speed is the other lower\-coverage case, with 27\.9% nulls, which likely reflects sparse or unavailable taxi\-speed observations in some local baseline groups\. The absolute z\-score averages are all in a reasonable range, while the largest z\-scores flag sharp local deviations that will be useful for anomaly\-severity scoring rather than final anomaly labeling\.

### Generate Percentile\-Rank Features

Next, we add percentile ranks\. These score each observation against the empirical history for the same Taxi Zone × Temporal Bucket × congestion\-pricing regime\. Unlike z\-scores, percentile ranks do not assume a roughly normal distribution, which makes them a useful companion feature for skewed mobility metrics\.

In [11]:
# ---------------------------------------------------------------------
# Generate percentile-rank anomaly-severity features
# ---------------------------------------------------------------------

percentile_start_time = time.perf_counter()

# The z-score block should already have created this skinny feature table.
# Keep using it instead of copying the full panel.
assert "anomaly_severity_feature_df" in globals(), (
    "Run the z-score anomaly-severity feature block before percentile ranks."
)

if "_row_id" not in anomaly_severity_feature_df.columns:
    anomaly_severity_feature_df["_row_id"] = np.arange(
        len(anomaly_severity_feature_df)
    )

percentile_feature_records = []

for metric in ANOMALY_SEVERITY_METRICS:
    metric_start_time = time.perf_counter()

    percentile_column = f"{metric}_deviation_percentile_rank"

    # Work with only the columns needed for this metric and preserve row order.
    metric_work_df = (
        anomaly_df[
            [
                "taxi_zone_id",
                "date",
                "temporal_bucket",
                "pre_post_cp",
                metric,
            ]
        ]
        .copy()
    )

    metric_work_df["_row_id"] = np.arange(len(metric_work_df))

    metric_reference_df = (
        percentile_reference_df
        .loc[
            percentile_reference_df["metric"] == metric,
            BASELINE_GROUP_COLUMNS
            + [
                "reference_observations",
                "percentile_reference_ready",
            ],
        ]
        .copy()
    )

    # Attach reference readiness so thin or unavailable histories remain null.
    metric_work_df = metric_work_df.merge(
        metric_reference_df,
        on=BASELINE_GROUP_COLUMNS,
        how="left",
        validate="many_to_one",
    )

    ready_mask = (
        metric_work_df["percentile_reference_ready"].fillna(False)
        & metric_work_df[metric].notna()
    )

    metric_work_df[percentile_column] = np.nan

    # Percentile ranks are calculated inside each local baseline group.
    metric_work_df.loc[ready_mask, percentile_column] = (
        metric_work_df
        .loc[ready_mask]
        .groupby(BASELINE_GROUP_COLUMNS, observed=True)[metric]
        .rank(method="average", pct=True)
    )

    # Add only the new percentile feature to the skinny feature table.
    anomaly_severity_feature_df[percentile_column] = (
        metric_work_df[percentile_column]
        .to_numpy()
    )

    percentile_feature_records.append(
        {
            "metric": metric,
            "ready_reference_groups": metric_reference_df[
                "percentile_reference_ready"
            ].sum(),
            "percentile_non_null_observations": metric_work_df[
                percentile_column
            ].notna().sum(),
            "percentile_null_pct": metric_work_df[
                percentile_column
            ].isna().mean(),
            "mean_percentile_rank": metric_work_df[
                percentile_column
            ].mean(),
            "min_percentile_rank": metric_work_df[
                percentile_column
            ].min(),
            "max_percentile_rank": metric_work_df[
                percentile_column
            ].max(),
            "runtime_seconds": time.perf_counter() - metric_start_time,
        }
    )

    print(
        f"Created percentile-rank feature for {metric} "
        f"in {time.perf_counter() - metric_start_time:.2f} seconds."
    )

    del metric_work_df, metric_reference_df, ready_mask
    gc.collect()

percentile_feature_summary_df = pd.DataFrame(percentile_feature_records)

percentile_feature_summary_df[
    [
        "percentile_null_pct",
        "mean_percentile_rank",
        "min_percentile_rank",
        "max_percentile_rank",
        "runtime_seconds",
    ]
] = (
    percentile_feature_summary_df[
        [
            "percentile_null_pct",
            "mean_percentile_rank",
            "min_percentile_rank",
            "max_percentile_rank",
            "runtime_seconds",
        ]
    ]
    .round(4)
)

display(percentile_feature_summary_df)

expected_percentile_columns = [
    f"{metric}_deviation_percentile_rank"
    for metric in ANOMALY_SEVERITY_METRICS
]

missing_percentile_columns = [
    column
    for column in expected_percentile_columns
    if column not in anomaly_severity_feature_df.columns
]

assert not missing_percentile_columns, (
    f"Missing generated percentile feature columns: {missing_percentile_columns}"
)

assert (
    percentile_feature_summary_df["percentile_non_null_observations"]
    .gt(0)
    .all()
), (
    "At least one metric produced no non-null percentile-rank values."
)

assert (
    anomaly_severity_feature_df[expected_percentile_columns]
    .max()
    .le(1)
    .all()
), (
    "One or more percentile-rank features contains values above 1."
)

assert (
    anomaly_severity_feature_df[expected_percentile_columns]
    .min()
    .ge(0)
    .all()
), (
    "One or more percentile-rank features contains values below 0."
)

print(
    "Percentile-rank anomaly-severity feature step complete. "
    f"Runtime: {time.perf_counter() - percentile_start_time:.2f} seconds."
)

Created percentile-rank feature for bus_trip_count in 0.93 seconds.
Created percentile-rank feature for avg_bus_speed in 0.94 seconds.
Created percentile-rank feature for subway_ridership in 0.79 seconds.
Created percentile-rank feature for taxi_trip_count in 0.90 seconds.
Created percentile-rank feature for taxi_avg_trip_speed in 0.93 seconds.
Created percentile-rank feature for fhvhv_trip_count in 0.99 seconds.
Created percentile-rank feature for fhvhv_avg_trip_speed in 1.12 seconds.


,metric,ready_reference_groups,percentile_non_null_observations,percentile_null_pct,mean_percentile_rank,min_percentile_rank,max_percentile_rank,runtime_seconds
0,bus_trip_count,4980,1476465,0.0533,0.5017,0.0048,1.0,0.9263
1,avg_bus_speed,4980,1476465,0.0533,0.5017,0.0019,1.0,0.9429
2,subway_ridership,3120,925080,0.4068,0.5017,0.0019,1.0,0.7889
3,taxi_trip_count,5260,1559590,0.0000,0.5017,0.0019,1.0,0.9034
4,taxi_avg_trip_speed,4964,1124054,0.2793,0.5022,0.0019,1.0,0.9333
5,fhvhv_trip_count,5260,1559590,0.0000,0.5017,0.0019,1.0,0.9933
6,fhvhv_avg_trip_speed,5260,1559590,0.0000,0.5017,0.0019,1.0,1.1155


Percentile-rank anomaly-severity feature step complete. Runtime: 7.37 seconds.


Findings\. The percentile\-rank step ran even faster than the z\-score step, completing all seven metrics in about 7 seconds\. The percentile features have the expected shape: values stay between 0 and 1, and the mean percentile rank sits right around 0\.50 for every metric, which is what we want from within\-group empirical ranking\. Coverage mostly follows the same baseline\-readiness pattern as before, but percentile ranks are slightly more available than z\-scores for count metrics because they do not require a nonzero standard deviation\. Taxi trip count and both FHVHV metrics are fully populated, Bus has about 5\.3% nulls, Taxi average trip speed has 27\.9% nulls, and Subway has the expected higher null rate given its smaller geographic footprint\.

### Section Summary

We generated the anomaly\-severity features for the seven core mobility metrics\. Each metric now has a local historical z\-score, absolute z\-score, and percentile rank calculated within Taxi Zone × Temporal Bucket × congestion\-pricing regime\. These features measure how unusual each mobility observation looks relative to its own local history, without assigning final anomaly labels or changing the panel grain\.

## 1\.5\.5\.4 Validate and Create Manifest

Now we run a compact QA pass on the anomaly\-severity feature table\. The goal is to confirm that all planned features were created, the original panel grain stayed intact, and the new feature values behave as expected\. Nulls are allowed where the source metric or local baseline is unavailable, but zero\-coverage features, duplicate grain rows, non\-finite values, or percentile ranks outside 0–1 would be blockers\.

### Validation

Validate Feature Inventory and Grain\. This block checks the basics: all planned anomaly\-severity features exist, the feature table still has the same number of rows as the input panel, and the Taxi Zone × Date × Temporal Bucket grain has not been duplicated\.

In [12]:
# ---------------------------------------------------------------------
# Validate anomaly feature inventory and grain
# ---------------------------------------------------------------------

expected_anomaly_feature_columns = ANOMALY_SEVERITY_FEATURE_COLUMNS

missing_anomaly_feature_columns = [
    column
    for column in expected_anomaly_feature_columns
    if column not in anomaly_severity_feature_df.columns
]

duplicate_anomaly_feature_grain_rows = (
    anomaly_severity_feature_df
    .duplicated(GRAIN_COLUMNS)
    .sum()
)

anomaly_feature_inventory_qa_df = pd.DataFrame(
    [
        {
            "qa_check": "input_panel_rows",
            "value": len(anomaly_df),
            "status": "reference",
        },
        {
            "qa_check": "anomaly_feature_rows",
            "value": len(anomaly_severity_feature_df),
            "status": (
                "pass"
                if len(anomaly_severity_feature_df) == len(anomaly_df)
                else "fail"
            ),
        },
        {
            "qa_check": "planned_anomaly_features",
            "value": len(expected_anomaly_feature_columns),
            "status": "reference",
        },
        {
            "qa_check": "generated_anomaly_features",
            "value": (
                len(expected_anomaly_feature_columns)
                - len(missing_anomaly_feature_columns)
            ),
            "status": (
                "pass"
                if len(missing_anomaly_feature_columns) == 0
                else "fail"
            ),
        },
        {
            "qa_check": "duplicate_grain_rows",
            "value": duplicate_anomaly_feature_grain_rows,
            "status": (
                "pass"
                if duplicate_anomaly_feature_grain_rows == 0
                else "fail"
            ),
        },
    ]
)

display(anomaly_feature_inventory_qa_df)

assert len(anomaly_severity_feature_df) == len(anomaly_df), (
    "Anomaly feature table row count does not match input panel."
)

assert not missing_anomaly_feature_columns, (
    f"Missing anomaly-severity features: {missing_anomaly_feature_columns}"
)

assert duplicate_anomaly_feature_grain_rows == 0, (
    "Anomaly feature table contains duplicate Taxi Zone × Date × Temporal Bucket rows."
)

,qa_check,value,status
0,input_panel_rows,1559590,reference
1,anomaly_feature_rows,1559590,pass
2,planned_anomaly_features,21,reference
3,generated_anomaly_features,21,pass
4,duplicate_grain_rows,0,pass


Findings\. The feature inventory check passed cleanly: the anomaly feature panel preserved all 1,559,590 rows, created all 21 planned features, and kept one row per Taxi Zone × Date × Temporal Bucket\.

Summarize Feature Coverage and Ranges\. This block summarizes null behavior and value ranges across the 21 new anomaly\-severity features\. The main things to watch are zero\-coverage features, out\-of\-range percentile ranks, and unusually high null rates that are not explained by source coverage\.

In [13]:
# ---------------------------------------------------------------------
# Summarize anomaly feature coverage and ranges
# ---------------------------------------------------------------------

anomaly_feature_coverage_records = []

for metric in ANOMALY_SEVERITY_METRICS:
    for feature_suffix in ANOMALY_SEVERITY_FEATURE_SUFFIXES:
        feature_name = f"{metric}_{feature_suffix}"
        feature_series = anomaly_severity_feature_df[feature_name]

        anomaly_feature_coverage_records.append(
            {
                "metric": metric,
                "feature_type": feature_suffix,
                "feature_name": feature_name,
                "non_null_observations": feature_series.notna().sum(),
                "null_observations": feature_series.isna().sum(),
                "null_pct": feature_series.isna().mean(),
                "min_value": feature_series.min(),
                "mean_value": feature_series.mean(),
                "max_value": feature_series.max(),
                "finite_values": np.isfinite(
                    feature_series.dropna()
                ).sum(),
            }
        )

anomaly_feature_coverage_df = pd.DataFrame(
    anomaly_feature_coverage_records
)

numeric_coverage_columns = [
    "null_pct",
    "min_value",
    "mean_value",
    "max_value",
]

anomaly_feature_coverage_df[numeric_coverage_columns] = (
    anomaly_feature_coverage_df[numeric_coverage_columns]
    .round(4)
)

display(anomaly_feature_coverage_df)

zero_coverage_features = (
    anomaly_feature_coverage_df["non_null_observations"]
    .eq(0)
    .sum()
)

nonfinite_feature_count = (
    anomaly_feature_coverage_df["finite_values"]
    .ne(anomaly_feature_coverage_df["non_null_observations"])
    .sum()
)

percentile_columns = [
    f"{metric}_deviation_percentile_rank"
    for metric in ANOMALY_SEVERITY_METRICS
]

percentile_below_zero_count = (
    anomaly_severity_feature_df[percentile_columns]
    .lt(0)
    .sum()
    .sum()
)

percentile_above_one_count = (
    anomaly_severity_feature_df[percentile_columns]
    .gt(1)
    .sum()
    .sum()
)

assert zero_coverage_features == 0, (
    "One or more anomaly-severity features has zero non-null observations."
)

assert nonfinite_feature_count == 0, (
    "One or more anomaly-severity features contains non-finite values."
)

assert percentile_below_zero_count == 0, (
    "One or more percentile-rank values is below 0."
)

assert percentile_above_one_count == 0, (
    "One or more percentile-rank values is above 1."
)

,metric,feature_type,feature_name,non_null_observations,null_observations,null_pct,min_value,mean_value,max_value,finite_values
0,bus_trip_count,deviation_zscore,bus_trip_count_deviation_zscore,1476448,83142,0.0533,-3.8775,0.0000,7.2295,1476448
1,bus_trip_count,deviation_abs_zscore,bus_trip_count_deviation_abs_zscore,1476448,83142,0.0533,0.0000,0.8640,7.2295,1476448
2,bus_trip_count,deviation_percentile_rank,bus_trip_count_deviation_percentile_rank,1476465,83125,0.0533,0.0048,0.5017,1.0000,1476465
3,avg_bus_speed,deviation_zscore,avg_bus_speed_deviation_zscore,1476465,83125,0.0533,-5.5972,-0.0000,8.2588,1476465
4,avg_bus_speed,deviation_abs_zscore,avg_bus_speed_deviation_abs_zscore,1476465,83125,0.0533,0.0000,0.8017,8.2588,1476465
5,avg_bus_speed,deviation_percentile_rank,avg_bus_speed_deviation_percentile_rank,1476465,83125,0.0533,0.0019,0.5017,1.0000,1476465
6,subway_ridership,deviation_zscore,subway_ridership_deviation_zscore,918315,641275,0.4112,-9.8532,-0.0000,22.2995,918315
7,subway_ridership,deviation_abs_zscore,subway_ridership_deviation_abs_zscore,918315,641275,0.4112,0.0000,0.7108,22.2995,918315
8,subway_ridership,deviation_percentile_rank,subway_ridership_deviation_percentile_rank,925080,634510,0.4068,0.0019,0.5017,1.0000,925080
9,taxi_trip_count,deviation_zscore,taxi_trip_count_deviation_zscore,1547183,12407,0.0080,-9.0108,-0.0000,22.8692,1547183


Findings\. Feature coverage looks healthy overall, with every feature populated somewhere, all values finite, percentile ranks staying inside 0–1, and the highest null rates matching expected metric availability patterns\.

Compact QA Summary\. This final QA table rolls the feature\-generation checks into a short reader\-friendly summary\. It gives us a clean handoff before moving into the redundancy/complementarity comparison against 1\.5\.4 residual features\.

In [14]:
# ---------------------------------------------------------------------
# Create compact anomaly feature QA summary
# ---------------------------------------------------------------------

anomaly_feature_qa_summary_df = pd.DataFrame(
    [
        {
            "qa_check": "row_count_preserved",
            "value": len(anomaly_severity_feature_df) == len(anomaly_df),
            "status": (
                "pass"
                if len(anomaly_severity_feature_df) == len(anomaly_df)
                else "fail"
            ),
        },
        {
            "qa_check": "grain_preserved",
            "value": duplicate_anomaly_feature_grain_rows == 0,
            "status": (
                "pass"
                if duplicate_anomaly_feature_grain_rows == 0
                else "fail"
            ),
        },
        {
            "qa_check": "planned_features_created",
            "value": len(expected_anomaly_feature_columns),
            "status": (
                "pass"
                if len(missing_anomaly_feature_columns) == 0
                else "fail"
            ),
        },
        {
            "qa_check": "features_with_zero_coverage",
            "value": zero_coverage_features,
            "status": (
                "pass"
                if zero_coverage_features == 0
                else "fail"
            ),
        },
        {
            "qa_check": "features_with_nonfinite_values",
            "value": nonfinite_feature_count,
            "status": (
                "pass"
                if nonfinite_feature_count == 0
                else "fail"
            ),
        },
        {
            "qa_check": "max_feature_null_pct",
            "value": anomaly_feature_coverage_df["null_pct"].max(),
            "status": "review",
        },
        {
            "qa_check": "percentile_values_below_zero",
            "value": percentile_below_zero_count,
            "status": (
                "pass"
                if percentile_below_zero_count == 0
                else "fail"
            ),
        },
        {
            "qa_check": "percentile_values_above_one",
            "value": percentile_above_one_count,
            "status": (
                "pass"
                if percentile_above_one_count == 0
                else "fail"
            ),
        },
    ]
)

display(anomaly_feature_qa_summary_df)

assert (
    anomaly_feature_qa_summary_df
    .loc[
        anomaly_feature_qa_summary_df["status"].isin(["pass", "fail"]),
        "status",
    ]
    .eq("pass")
    .all()
), (
    "One or more anomaly feature QA checks failed."
)

print(
    "Anomaly feature QA summary complete. "
    f"Validated {len(expected_anomaly_feature_columns):,} generated features."
)

,qa_check,value,status
0,row_count_preserved,True,pass
1,grain_preserved,True,pass
2,planned_features_created,21,pass
3,features_with_zero_coverage,0,pass
4,features_with_nonfinite_values,0,pass
5,max_feature_null_pct,0.4112,review
6,percentile_values_below_zero,0,pass
7,percentile_values_above_one,0,pass


Anomaly feature QA summary complete. Validated 21 generated features.


Findings\. The compact QA summary passed every blocker check, with only the 41\.1% maximum null rate marked for review because some source metrics naturally have partial panel coverage\.

### Create Manifest

We document the new anomaly\-severity features so the final output has clear names, definitions, baseline logic, and null behavior before the panel gets written to disk\.

In [15]:
# ---------------------------------------------------------------------
# Create anomaly-severity feature manifest
# ---------------------------------------------------------------------

feature_type_definitions = {
    "deviation_zscore": {
        "display_name": "Deviation z-score",
        "definition_template": (
            "{metric} minus its local historical mean, divided by the local "
            "historical standard deviation."
        ),
        "interpretation": (
            "Larger positive or negative values indicate observations farther "
            "from the local historical baseline."
        ),
        "null_behavior": (
            "Null when the source metric is null, the local baseline is not ready, "
            "or the local baseline standard deviation is zero or unavailable."
        ),
    },
    "deviation_abs_zscore": {
        "display_name": "Absolute deviation z-score",
        "definition_template": (
            "Absolute value of the {metric} local historical deviation z-score."
        ),
        "interpretation": (
            "Larger values indicate more unusual observations, regardless of direction."
        ),
        "null_behavior": (
            "Null when the corresponding deviation z-score is null."
        ),
    },
    "deviation_percentile_rank": {
        "display_name": "Deviation percentile rank",
        "definition_template": (
            "Empirical percentile rank of {metric} within its local historical "
            "baseline group."
        ),
        "interpretation": (
            "Values near 0 are low relative to local history; values near 1 are high "
            "relative to local history."
        ),
        "null_behavior": (
            "Null when the source metric is null or the local percentile reference "
            "group is not ready."
        ),
    },
}

manifest_records = []

for metric in ANOMALY_SEVERITY_METRICS:
    for feature_suffix in ANOMALY_SEVERITY_FEATURE_SUFFIXES:
        feature_name = f"{metric}_{feature_suffix}"
        feature_info = feature_type_definitions[feature_suffix]

        # Keep each feature definition explicit enough for downstream notebooks
        # without turning the manifest into a long methods section.
        manifest_records.append(
            {
                "feature_name": feature_name,
                "source_metric": metric,
                "feature_family": "anomaly_severity",
                "feature_type": feature_suffix,
                "feature_display_name": feature_info["display_name"],
                "definition": feature_info["definition_template"].format(
                    metric=metric
                ),
                "baseline_grain": " × ".join(BASELINE_GROUP_COLUMNS),
                "baseline_type": "full-history retrospective",
                "congestion_pricing_handling": (
                    "Baselines are separated by congestion-pricing regime."
                ),
                "interpretation": feature_info["interpretation"],
                "null_behavior": feature_info["null_behavior"],
                "production_status": "production",
                "created_in_notebook": "1.5.5",
                "input_panel": str(INPUT_PANEL_PATH),
                "output_panel": str(OUTPUT_PANEL_PATH),
            }
        )

variability_anomaly_feature_manifest_df = pd.DataFrame(manifest_records)

manifest_column_order = [
    "feature_name",
    "source_metric",
    "feature_family",
    "feature_type",
    "feature_display_name",
    "definition",
    "baseline_grain",
    "baseline_type",
    "congestion_pricing_handling",
    "interpretation",
    "null_behavior",
    "production_status",
    "created_in_notebook",
    "input_panel",
    "output_panel",
]

variability_anomaly_feature_manifest_df = (
    variability_anomaly_feature_manifest_df[manifest_column_order]
)

display(variability_anomaly_feature_manifest_df)

assert len(variability_anomaly_feature_manifest_df) == len(
    ANOMALY_SEVERITY_FEATURE_COLUMNS
), (
    "Manifest row count does not match planned anomaly-severity feature count."
)

assert set(variability_anomaly_feature_manifest_df["feature_name"]) == set(
    ANOMALY_SEVERITY_FEATURE_COLUMNS
), (
    "Manifest feature names do not match generated anomaly-severity features."
)

assert (
    variability_anomaly_feature_manifest_df["feature_name"]
    .is_unique
), (
    "Manifest contains duplicate feature names."
)


,feature_name,source_metric,feature_family,feature_type,feature_display_name,definition,baseline_grain,baseline_type,congestion_pricing_handling,interpretation,null_behavior,production_status,created_in_notebook,input_panel,output_panel
0,bus_trip_count_deviation_zscore,bus_trip_count,anomaly_severity,deviation_zscore,Deviation z-score,bus_trip_count minus its local historical mean...,taxi_zone_id × temporal_bucket × pre_post_cp,full-history retrospective,Baselines are separated by congestion-pricing ...,Larger positive or negative values indicate ob...,"Null when the source metric is null, the local...",production,1.5.5,pipeline_data/1.5.4.final_tables/temporal_deco...,pipeline_data/1.5.5.final_tables/variability_a...
1,bus_trip_count_deviation_abs_zscore,bus_trip_count,anomaly_severity,deviation_abs_zscore,Absolute deviation z-score,Absolute value of the bus_trip_count local his...,taxi_zone_id × temporal_bucket × pre_post_cp,full-history retrospective,Baselines are separated by congestion-pricing ...,Larger values indicate more unusual observatio...,Null when the corresponding deviation z-score ...,production,1.5.5,pipeline_data/1.5.4.final_tables/temporal_deco...,pipeline_data/1.5.5.final_tables/variability_a...
2,bus_trip_count_deviation_percentile_rank,bus_trip_count,anomaly_severity,deviation_percentile_rank,Deviation percentile rank,Empirical percentile rank of bus_trip_count wi...,taxi_zone_id × temporal_bucket × pre_post_cp,full-history retrospective,Baselines are separated by congestion-pricing ...,Values near 0 are low relative to local histor...,Null when the source metric is null or the loc...,production,1.5.5,pipeline_data/1.5.4.final_tables/temporal_deco...,pipeline_data/1.5.5.final_tables/variability_a...
3,avg_bus_speed_deviation_zscore,avg_bus_speed,anomaly_severity,deviation_zscore,Deviation z-score,"avg_bus_speed minus its local historical mean,...",taxi_zone_id × temporal_bucket × pre_post_cp,full-history retrospective,Baselines are separated by congestion-pricing ...,Larger positive or negative values indicate ob...,"Null when the source metric is null, the local...",production,1.5.5,pipeline_data/1.5.4.final_tables/temporal_deco...,pipeline_data/1.5.5.final_tables/variability_a...
4,avg_bus_speed_deviation_abs_zscore,avg_bus_speed,anomaly_severity,deviation_abs_zscore,Absolute deviation z-score,Absolute value of the avg_bus_speed local hist...,taxi_zone_id × temporal_bucket × pre_post_cp,full-history retrospective,Baselines are separated by congestion-pricing ...,Larger values indicate more unusual observatio...,Null when the corresponding deviation z-score ...,production,1.5.5,pipeline_data/1.5.4.final_tables/temporal_deco...,pipeline_data/1.5.5.final_tables/variability_a...
5,avg_bus_speed_deviation_percentile_rank,avg_bus_speed,anomaly_severity,deviation_percentile_rank,Deviation percentile rank,Empirical percentile rank of avg_bus_speed wit...,taxi_zone_id × temporal_bucket × pre_post_cp,full-history retrospective,Baselines are separated by congestion-pricing ...,Values near 0 are low relative to local histor...,Null when the source metric is null or the loc...,production,1.5.5,pipeline_data/1.5.4.final_tables/temporal_deco...,pipeline_data/1.5.5.final_tables/variability_a...
6,subway_ridership_deviation_zscore,subway_ridership,anomaly_severity,deviation_zscore,Deviation z-score,subway_ridership minus its local historical me...,taxi_zone_id × temporal_bucket × pre_post_cp,full-history retrospective,Baselines are separated by congestion-pricing ...,Larger positive or negative values indicate ob...,"Null when the source metric is null, the local...",production,1.5.5,pipeline_data/1.5.4.final_tables/temporal_deco...,pipeline_data/1.5.5.final_tables/variability_a...
7,subway_ridership_deviation_abs_zscore,subway_ridership,anomaly_severity,deviation_abs_zscore,Absolute deviation z-score,Absolute value of the subway_ridership local h...,taxi_zone_id × temporal_bucket × pre_post_cp,full-history retrospecti

### Data Dictionary

Anomaly\-severity features

<base\_metric\>\_deviation\_zscore\. Local historical z\-score for the observed metric, calculated within Taxi Zone × Temporal Bucket × congestion\-pricing regime\. Positive values mean the observation is above its local historical baseline; negative values mean it is below\.

<base\_metric\>\_deviation\_abs\_zscore\. Absolute value of the local historical z\-score\. Higher values indicate more unusual mobility behavior regardless of whether the observation is unusually high or unusually low\.

<base\_metric\>\_deviation\_percentile\_rank\. Empirical percentile rank of the observed metric within its local historical baseline group\. Values near 0 indicate unusually low observations; values near 1 indicate unusually high observations\.

Base metrics

base\_metric — One of: bus\_trip\_count, avg\_bus\_speed, subway\_ridership, taxi\_trip\_count, taxi\_avg\_trip\_speed, fhvhv\_trip\_count, fhvhv\_avg\_trip\_speed\.

Null behavior

Anomaly\-severity features may be null when the source metric is null, when a local baseline group does not meet the minimum observation threshold, or when z\-score features cannot be calculated because the local baseline standard deviation is zero or unavailable\. Percentile ranks may be available in some cases where z\-scores are not, because they do not require a nonzero standard deviation\.

### Section Summary

We validated the completed anomaly\-severity feature table and documented the new feature family for downstream use\. The QA checks confirm that the panel row count and Taxi Zone × Date × Temporal Bucket grain were preserved, all 21 planned features were created, feature values are finite where populated, and percentile ranks stay within the expected 0–1 range\. The manifest and data dictionary now record each feature’s source metric, baseline grain, interpretation, and null behavior before the final outputs are written\.

## 1\.5\.5\.5 Write Outputs

We write the completed anomaly\-severity feature panel, manifest, and QA summary to the 1\.5\.5 final output directory\.

In [16]:
# ---------------------------------------------------------------------
# Write final 1.5.5 outputs
# ---------------------------------------------------------------------

assert "anomaly_severity_feature_df" in globals(), (
    "Run anomaly-severity feature generation before writing final outputs."
)

assert "variability_anomaly_feature_manifest_df" in globals(), (
    "Run the manifest block before writing final outputs."
)

assert "anomaly_feature_qa_summary_df" in globals(), (
    "Run the compact QA summary block before writing final outputs."
)


def duckdb_path(path):
    """Convert a pathlib path to a DuckDB-friendly string."""
    return str(path).replace("\\", "/")


final_anomaly_feature_columns = [
    column
    for column in ANOMALY_SEVERITY_FEATURE_COLUMNS
    if column in anomaly_severity_feature_df.columns
]

missing_final_anomaly_features = [
    column
    for column in ANOMALY_SEVERITY_FEATURE_COLUMNS
    if column not in final_anomaly_feature_columns
]

assert not missing_final_anomaly_features, (
    f"Missing anomaly-severity features before write: {missing_final_anomaly_features}"
)

# Keep the registered feature table skinny so the final join does not copy
# the full 1.5.4 panel in memory.
anomaly_feature_write_df = anomaly_severity_feature_df[
    GRAIN_COLUMNS + final_anomaly_feature_columns
].copy()

con = duckdb.connect()
con.register("anomaly_feature_write_df", anomaly_feature_write_df)

feature_select_sql = ",\n                ".join(
    [
        f"features.{feature_name}"
        for feature_name in final_anomaly_feature_columns
    ]
)

if WRITE_OUTPUTS:
    if OUTPUT_PANEL_PATH.exists():
        OUTPUT_PANEL_PATH.unlink()

    # Stream the original 1.5.4 panel from parquet and join only the new
    # skinny feature table. This avoids holding the full base panel plus all
    # new features in pandas during the final write.
    con.sql(
        f"""
        COPY (
            SELECT
                base.*,
                {feature_select_sql}
            FROM read_parquet('{duckdb_path(INPUT_PANEL_PATH)}') AS base
            LEFT JOIN anomaly_feature_write_df AS features
                USING (taxi_zone_id, date, temporal_bucket)
        )
        TO '{duckdb_path(OUTPUT_PANEL_PATH)}'
        (FORMAT PARQUET)
        """
    )

    variability_anomaly_feature_manifest_df.to_csv(
        OUTPUT_MANIFEST_PATH,
        index=False,
    )

    anomaly_feature_qa_summary_df.to_csv(
        OUTPUT_QA_PATH,
        index=False,
    )

output_row_count = con.sql(
    f"""
    SELECT COUNT(*)
    FROM read_parquet('{duckdb_path(OUTPUT_PANEL_PATH)}')
    """
).fetchone()[0]

output_schema_df = con.sql(
    f"""
    DESCRIBE SELECT *
    FROM read_parquet('{duckdb_path(OUTPUT_PANEL_PATH)}')
    """
).df()

output_columns = set(output_schema_df["column_name"])

missing_written_features = [
    column
    for column in final_anomaly_feature_columns
    if column not in output_columns
]

output_write_summary_df = pd.DataFrame(
    [
        {
            "output_name": "variability_anomaly_feature_panel",
            "path": str(OUTPUT_PANEL_PATH),
            "exists": OUTPUT_PANEL_PATH.exists(),
            "rows": output_row_count,
            "columns": len(output_columns),
        },
        {
            "output_name": "variability_anomaly_feature_manifest",
            "path": str(OUTPUT_MANIFEST_PATH),
            "exists": OUTPUT_MANIFEST_PATH.exists(),
            "rows": len(variability_anomaly_feature_manifest_df),
            "columns": variability_anomaly_feature_manifest_df.shape[1],
        },
        {
            "output_name": "variability_anomaly_feature_qa_summary",
            "path": str(OUTPUT_QA_PATH),
            "exists": OUTPUT_QA_PATH.exists(),
            "rows": len(anomaly_feature_qa_summary_df),
            "columns": anomaly_feature_qa_summary_df.shape[1],
        },
    ]
)

display(output_write_summary_df)

assert output_row_count == len(anomaly_df), (
    "Final 1.5.5 output panel row count does not match input panel."
)

assert not missing_written_features, (
    f"Final 1.5.5 output panel is missing features: {missing_written_features}"
)

assert OUTPUT_PANEL_PATH.exists(), "Final 1.5.5 panel was not written."
assert OUTPUT_MANIFEST_PATH.exists(), "Final 1.5.5 manifest was not written."
assert OUTPUT_QA_PATH.exists(), "Final 1.5.5 QA summary was not written."

con.close()
del anomaly_feature_write_df
gc.collect()

print("Final 1.5.5 outputs written successfully.")

,output_name,path,exists,rows,columns
0,variability_anomaly_feature_panel,pipeline_data/1.5.5.final_tables/variability_a...,True,1559590,342
1,variability_anomaly_feature_manifest,pipeline_data/1.5.5.final_tables/variability_a...,True,21,15
2,variability_anomaly_feature_qa_summary,pipeline_data/1.5.5.final_tables/variability_a...,True,8,3


Final 1.5.5 outputs written successfully.


In [ ]:
# ---------------------------------------------------------------------
# Cleanup notebook memory
# ---------------------------------------------------------------------

import gc

large_objects_to_delete = [
    "anomaly_df",
    "anomaly_severity_feature_df",
    "baseline_statistics_df",
    "baseline_statistics_summary_df",
    "baseline_statistics_preview_df",
    "percentile_reference_df",
    "percentile_reference_summary_df",
    "zscore_anomaly_feature_df",
    "percentile_anomaly_feature_df",
    "anomaly_feature_coverage_df",
    "anomaly_feature_qa_summary_df",
    "variability_anomaly_feature_manifest_df",
    "output_write_summary_df",
    "output_schema_df",
]

for object_name in large_objects_to_delete:
    if object_name in globals():
        del globals()[object_name]

gc.collect()

print("1.5.5 memory cleanup complete.")

## Close

This notebook adds anomaly\-severity features to the completed decomposition panel without changing the project grain\. The workflow loads the 1\.5\.4 panel, builds local historical baselines by Taxi Zone × Temporal Bucket × congestion\-pricing regime, generates z\-score, absolute z\-score, and percentile\-rank features for the seven core mobility metrics, validates feature coverage and grain preservation, and documents the new feature family with a manifest and data dictionary\. The finished output gives downstream notebooks a compact set of variability\-aware features for clustering, density estimation, and outlier detection while leaving final feature selection and scaling decisions to 1\.5\.6\.

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=4a322346-8e1e-4650-8cef-fe9b767d96fb' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>